In [2]:
import pandas as pd
from scipy.sparse import coo_matrix
import implicit

# Path to your CSV file
path_to_csv = '../datasets/msd_kaggle/User Listening History.csv'

# Read the CSV file into a pandas DataFrame
data = pd.read_csv(path_to_csv)

# print("Data sample:")
# print(data.head())

# Assuming your CSV has columns: index, track_id, user_id, playcount,
# we use 'track_id' for items (songs) and 'user_id' for users.

# Create mappings from original IDs to indices for users and tracks
user_ids = data['user_id'].unique()
track_ids = data['track_id'].unique()

user_to_idx = {user: idx for idx, user in enumerate(user_ids)}
track_to_idx = {track: idx for idx, track in enumerate(track_ids)}

# Map original ids to indices
data['user_idx'] = data['user_id'].map(user_to_idx)
data['track_idx'] = data['track_id'].map(track_to_idx)

# Build a sparse matrix in COO format
# Here, rows correspond to tracks (items) and columns correspond to users.
sparse_matrix = coo_matrix(
    (data['playcount'], (data['track_idx'], data['user_idx'])),
    shape=(len(track_ids), len(user_ids))
)

# Convert to CSR format for efficiency
sparse_matrix = sparse_matrix.tocsr()

alpha = 40
confidence = sparse_matrix * alpha
confidence_transpose = confidence.T


In [ ]:
# model
model = implicit.als.AlternatingLeastSquares(
    factors=100,
    regularization=0.01,
    iterations=50,
    use_gpu=False  # Set to True if you have a compatible GPU
)

model.fit(confidence_transpose) # for future reference, this is the model that we will use

# Retrieve the latent factors for tracks and users 
track_factors = model.item_factors
user_factors = model.user_factors

100%|██████████| 50/50 [05:49<00:00,  6.99s/it]


In [5]:
print(type(user_factors))
print(user_factors.shape)

<class 'numpy.ndarray'>
(30459, 100)


In [7]:
import pandas as pd

# Assuming that after transposition:
# model.user_factors -> latent factors for tracks, shape should be (30459, factors)
# model.item_factors -> latent factors for users, shape should be (962037, factors)

# Save track (item) latent factors
df_tracks = pd.DataFrame(model.user_factors, index=list(track_ids))
df_tracks.index.name = 'track_id'
df_tracks.to_csv('track_factors.csv')
print("Track factors saved to track_factors.csv")

# Save user latent factors
df_users = pd.DataFrame(model.item_factors, index=list(user_ids))
df_users.index.name = 'user_id'
df_users.to_csv('user_factors.csv')
print("User factors saved to user_factors.csv")

Track factors saved to track_factors.csv
User factors saved to user_factors.csv


Let us generate avg latent factors for each genre.

In [29]:
import pandas as pd

# Read the latent factors file.
df_factors = pd.read_csv('track_factors.csv')
print("Track factors data sample:")
print(df_factors.head())

# Read the music info file.
df_info = pd.read_csv('../datasets/msd_kaggle/Music Info.csv')
print("Music info data sample:")
print(df_info.head())

# Merge the two DataFrames on 'track_id'
df_merged = pd.merge(df_factors, df_info[['track_id', 'tags']], on='track_id', how='inner')

# Fill missing tags with empty strings
df_merged['tags'] = df_merged['tags'].fillna('')

# Split the comma-separated tags into a list of genres for each track.
df_merged['genre_list'] = df_merged['tags'].apply(
    lambda x: [tag.strip() for tag in x.split(',') if tag.strip()]
)

# Explode the genre_list so that each row corresponds to one genre for a track.
df_exploded = df_merged.explode('genre_list').rename(columns={'genre_list': 'genre'})

# Identify latent factor columns. Assuming latent factor columns are all columns in df_factors except 'track_id'.
latent_cols = [col for col in df_factors.columns if col != 'track_id']

# Group by genre and compute the mean latent factor across all tracks for each genre.
genre_avg = df_exploded.groupby('genre')[latent_cols].mean().reset_index()

# Save the resulting DataFrame to a CSV file.
genre_avg.to_csv('genre_avg_latent_factors.csv', index=False)
print("Average latent factors for each genre saved to genre_avg_latent_factors.csv")
print("Average latent factors per genre:")
print(genre_avg.head())

Track factors data sample:
             track_id         0         1         2         3         4  \
0  TRIRLYL128F42539D1  0.808989  2.290100 -0.647640 -2.801239  0.861788   
1  TRFUPBA128F934F7E1  1.512314 -1.623835 -0.164228  0.851826 -1.273279   
2  TRLQPQJ128F42AA94F  1.916364 -0.036905  1.416710 -3.117868  0.936974   
3  TRTUCUY128F92E1D24 -1.858678  0.021146  1.372160  0.297119 -4.627149   
4  TRHDDQG12903CB53EE  2.558880  0.641069  0.841406 -0.163380  2.108576   

          5         6         7         8  ...        90        91        92  \
0 -2.594983  1.033361  3.328980 -1.724754  ... -0.592306 -2.305194 -0.134340   
1 -0.153443 -5.257395 -3.173786  1.943965  ...  2.251638  3.289549  4.304942   
2 -0.556644  0.661530 -1.634735  0.397131  ...  2.160876  0.668993  0.649326   
3  0.185463  1.397052  0.164689  0.214657  ... -1.412308  1.640929 -0.110023   
4 -2.002474 -1.237024 -1.552683 -1.067473  ... -1.290653 -0.285162 -1.601201   

         93        94        95        96

In [37]:
df = pd.read_csv('genre_avg_latent_factors.csv')

# Access the latent factors for a specific genre (e.g., "rock")
genre_name = "rock"
genre_row = df.loc[df["genre"] == genre_name]

# Display the resulting row
print("Latent factors for", genre_name)
print(len(genre_row.columns))

Latent factors for rock
101
